In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    Trainer,
    TrainingArguments
)


import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

import proj_consts_
# from proj_consts_ import *

# easy conversion between pitch and int token, and finger and int token
# print("pitch_to_int_mapping: ", finger_mappings.pitch_to_int_mapping)
# print("int_to_pitch_mapping: ", finger_mappings.int_to_pitch_mapping)

# print("finger_to_int_mapping: ", finger_mappings.finger_to_int_mapping)
# print("int_to_finger_mapping: ", finger_mappings.int_to_finger_mapping)

pitch_to_int_mapping = proj_consts_.pitch_to_int_mapping
int_to_pitch_mapping = proj_consts_.int_to_pitch_mapping
finger_to_int_mapping = proj_consts_.finger_to_int_mapping
int_to_finger_mapping = proj_consts_.int_to_finger_mapping

In [ ]:
!unzip PianoFingeringDataset_v1.2.zip
!unzip ThumbSet_v1.0.zip

Streaming output truncated to the last 5000 lines.
  inflating: ThumbSet_v1.0/FingeringFiles/000000000607230510018-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000562338910011-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000602328100586-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000607721900087-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000630833900361-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000630750910968-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000642294300147-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000603193110038-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000636962400412-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000622180815602-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/000000000643595600337-1_fingering.txt  
  inflating: ThumbSet_v1.0/FingeringFiles/00000

In [ ]:
def convert_features_to_text(x_list):
    """
    Converts numeric features into a textual sequence for each note:
    [pitch_int, onset_time, offset_time, onset_vel, offset_vel, channel] -> "<p42> <on8> ... "
    Then we join them with spaces so each note becomes one token, e.g. "<p142> <on8> <off9> <vel80> <c0>"
    """
    tokens = []
    for row in x_list:
        pitch_int    = row[0]
        onset_time   = row[1]
        offset_time  = row[2]
        onset_vel    = row[3]
        offset_vel   = row[4]
        channel      = row[5]

        # Create a compact textual representation for each note
        token = f"<p{pitch_int}> <on{int(onset_time)}> <off{int(offset_time)}> <vel{int(onset_vel)}> <c{int(channel)}>"
        tokens.append(token)

    # We’ll just join with a special delimiter so each note is a separate "word".
    # (We will split again in the Dataset)
    return " | ".join(tokens)

def convert_fingers_to_text(y_list):
    """
    Converts finger labels (e.g. [1,2,3]) into a space-separated string: "1 2 3".
    We'll keep it for reference, though for token classification
    we typically need a parallel list, not just a big string.
    """
    return " ".join(str(f) for f in y_list)

In [ ]:
class BertFingeringDataset(Dataset):
    def __init__(self, sequences, tokenizer, max_length=128):
        """
        sequences: list of (X_list, y_list)
           where X_list is [[pitch_int, onset_time, ...], ...],
                 y_list is [finger1, finger2, ...]
        tokenizer: BertTokenizerFast
        max_length: max sequence length
        """
        self.sequences = sequences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        X_list, y_list = self.sequences[idx]

        # 1) Convert numeric arrays to a single text with " | " delimiter
        #    so each note is chunked into a single "word" (roughly).
        text_block = convert_features_to_text(X_list)
        # example => "<p142> <on8> <off9> <vel80> <c0> | <p136> <on9>..."

        # 2) Split into "words" by " | "
        #    Each 'word' corresponds to one note => y_list has a label for it
        note_tokens = text_block.split(" | ")  # array of note-level tokens

        # 3) Tokenize using "is_split_into_words=True" so BERT can align word IDs
        encoding = self.tokenizer(
            note_tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length"  # or "longest"
        )

        # 4) Align labels
        #    - "word_ids" maps each subword token back to its "word" index
        word_ids = encoding.word_ids()
        labels = []
        for word_id in word_ids:
            if word_id is None:
                # Padding or special tokens => ignore with -100
                labels.append(-100)
            else:
                # Use the finger from y_list
                labels.append(y_list[word_id])

        encoding["labels"] = labels

        # Convert everything to torch tensors
        # (Default data collator is fine if each sample is a dict of tensors)
        item = {k: torch.tensor(v) for k, v in encoding.items()}
        return item


# PRETRAINING

In [ ]:
import os
import pandas as pd

def load_encoded_sequences(folder_path, pitch_to_int_mapping, finger_to_int_mapping):
    """
    Loads all fingering files from a given folder_path,
    encodes them into (X_list, y_list) pairs, and returns a list of these pairs.
    """
    # Collect DataFrames
    all_dataframes = []

    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)

        # e.g. "001-01_fingering.txt" => "001-01", you might need a safer split if filenames differ
        fingering_label, _ = filename.split('_')  # "001-01"

        # If you need piece_id, fingering_type separately:
        # piece_id, fingering_type = fingering_label.split('-')

        # Check if this is a valid file
        if os.path.isfile(file_path):
            df = pd.read_table(
                file_path,
                sep="\t",
                skiprows=1,
                names=[
                    "noteID", "onset_time", "offset_time", "spelled_pitch",
                    "onset_velocity", "offset_velocity", "channel", "finger_number"
                ]
            )
            all_dataframes.append(df)

    print(f"Found {len(all_dataframes)} fingering files in {folder_path}")

    # Now encode each DataFrame into (X_list, y_list) pairs
    raw_encoded_sequences = []
    for df in all_dataframes:
        X_list = []
        y_list = []
        for row in df.itertuples(index=False):
            spelled_pitch = row.spelled_pitch
            finger_str    = str(row.finger_number)

            # Convert spelled pitch and finger to integer tokens
            pitch_int  = pitch_to_int_mapping.get(spelled_pitch, 0)
            finger_int = finger_to_int_mapping.get(finger_str, 0)

            # Feature row: (pitch, onset_time, offset_time, onset_vel, offset_vel, channel)
            feature_row = [
                pitch_int,
                float(row.onset_time),
                float(row.offset_time),
                float(row.onset_velocity),
                float(row.offset_velocity),
                float(row.channel)
            ]
            X_list.append(feature_row)
            y_list.append(finger_int)

        raw_encoded_sequences.append((X_list, y_list))

    return raw_encoded_sequences


In [ ]:
# Specify folder paths
PATH_TO_NOISY_SEQUENCES = './ThumbSet_v1.0/FingeringFiles'
PATH_TO_CLEAN_SEQUENCES = './PianoFingeringDataset_v1.2/FingeringFiles'

# Then, call the helper function to get the sequences
noisy_sequences = load_encoded_sequences(
    PATH_TO_NOISY_SEQUENCES,
    pitch_to_int_mapping,
    finger_to_int_mapping
)

high_quality_sequences = load_encoded_sequences(
    PATH_TO_CLEAN_SEQUENCES,
    pitch_to_int_mapping,
    finger_to_int_mapping
)

print(f"Noisy sequences: {len(noisy_sequences)}")
print(f"High quality sequences: {len(high_quality_sequences)}")


Found 178737 fingering files in ./ThumbSet_v1.0/FingeringFiles
Found 309 fingering files in ./PianoFingeringDataset_v1.2/FingeringFiles
Noisy sequences: 178737
High quality sequences: 309


In [ ]:
print(noisy_sequences[0])

([[142, 8.666666666666666, 9.333333333333332, 80.0, 80.0, 0.0], [136, 9.333333333333332, 9.999999999999998, 80.0, 80.0, 0.0], [145, 10.0, 10.666666666666666, 80.0, 80.0, 0.0], [142, 10.666666666666666, 11.333333333333332, 80.0, 80.0, 0.0], [136, 11.333333333333332, 11.999999999999998, 80.0, 80.0, 0.0], [145, 12.0, 12.666666666666666, 80.0, 80.0, 0.0], [139, 12.666666666666666, 13.333333333333332, 80.0, 80.0, 0.0], [133, 13.333333333333332, 13.999999999999998, 80.0, 80.0, 0.0], [145, 14.0, 14.666666666666666, 80.0, 80.0, 0.0], [139, 14.666666666666666, 15.333333333333332, 80.0, 80.0, 0.0], [133, 15.333333333333332, 15.999999999999998, 80.0, 80.0, 0.0], [142, 16.0, 16.666666666666668, 80.0, 80.0, 0.0], [139, 16.666666666666664, 17.333333333333332, 80.0, 80.0, 0.0], [136, 17.333333333333332, 18.0, 80.0, 80.0, 0.0], [142, 18.0, 18.666666666666668, 80.0, 80.0, 0.0], [139, 18.666666666666664, 19.33333333333333, 80.0, 80.0, 0.0], [136, 19.33333333333333, 20.0, 80.0, 80.0, 0.0], [139, 20.0, 20

In [ ]:
import pickle

# Save NOISY SEQS AS Pickle FOR FASTER PROCESSING LATER
with open("noisy_sequences.pkl", "wb") as f:
    pickle.dump(noisy_sequences, f)

In [ ]:
# # Convert all examples in noisy_sequences
# processed_noisy_sequences = [
#     (convert_features_to_text(X_list), convert_fingers_to_text(y_list))
#     for X_list, y_list in noisy_sequences
# ]

# # Convert all examples in high_quality_sequences
# processed_high_quality_sequences = [
#     (convert_features_to_text(X_list), convert_fingers_to_text(y_list))
#     for X_list, y_list in high_quality_sequences
# ]

# print(f"Processed {len(processed_noisy_sequences)} noisy sequences.")
# print(f"Processed {len(processed_high_quality_sequences)} high-quality sequences.")
# print(processed_noisy_sequences[0])

Processed 178737 noisy sequences.
Processed 309 high-quality sequences.
('<p142> <on8> <off9> <vel80> <c0> <p136> <on9> <off9> <vel80> <c0> <p145> <on10> <off10> <vel80> <c0> <p142> <on10> <off11> <vel80> <c0> <p136> <on11> <off11> <vel80> <c0> <p145> <on12> <off12> <vel80> <c0> <p139> <on12> <off13> <vel80> <c0> <p133> <on13> <off13> <vel80> <c0> <p145> <on14> <off14> <vel80> <c0> <p139> <on14> <off15> <vel80> <c0> <p133> <on15> <off15> <vel80> <c0> <p142> <on16> <off16> <vel80> <c0> <p139> <on16> <off17> <vel80> <c0> <p136> <on17> <off18> <vel80> <c0> <p142> <on18> <off18> <vel80> <c0> <p139> <on18> <off19> <vel80> <c0> <p136> <on19> <off20> <vel80> <c0> <p139> <on20> <off20> <vel80> <c0> <p136> <on20> <off21> <vel80> <c0> <p133> <on21> <off22> <vel80> <c0> <p139> <on22> <off22> <vel80> <c0> <p136> <on22> <off23> <vel80> <c0> <p133> <on23> <off24> <vel80> <c0> <p145> <on24> <off24> <vel80> <c0> <p142> <on24> <off25> <vel80> <c0> <p139> <on25> <off26> <vel80> <c0> <p142> <on26> <off26

# Pretraining Loop

In [ ]:
# Example:
# noisy_sequences  -> your 2,500 "noisy" pieces
# high_quality_sequences -> your 300+ "clean" pieces






from sklearn.model_selection import train_test_split

train_noisy, val_noisy = train_test_split(noisy_sequences, test_size=0.1, random_state=42)   # RANDOM STATE IS THE SPECIFIC SEED
print(f"num noisy seqs: {len(noisy_sequences)}")


bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
train_noisy_dataset = BertFingeringDataset(train_noisy, tokenizer=bert_tokenizer, max_length=256)
val_noisy_dataset   = BertFingeringDataset(val_noisy,   tokenizer=bert_tokenizer, max_length=256)

print("Train sample:", train_noisy_dataset[0])
print("Validation sample:", val_noisy_dataset[0])
print("Dataset sample shape =>", {k: v.shape for k, v in train_noisy_dataset[0].items()})




from transformers import BertForTokenClassification

num_finger_labels = len(finger_to_int_mapping)  # or however many finger classes you have

model = BertForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_finger_labels
)


train_args = TrainingArguments(
    output_dir="bert_noisy_pretrain",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=16,   # Increase batch size
    per_device_eval_batch_size=16,    # Increase batch size
    gradient_accumulation_steps=2,    # Helps when batch size is limited
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.01,
    fp16=True,  # Use mixed precision for speed
    optim="adamw_torch_fused",  # Faster optimizer
    dataloader_num_workers=4,  # Enable parallel data loading
    torch_compile=True,  # Try PyTorch 2.0 compilation for speed (optional)
    push_to_hub=False,
    report_to="none",  # disables wandb
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_noisy_dataset,
    eval_dataset=val_noisy_dataset,
    # No custom data collator => uses default DataCollatorWithPadding
)

trainer.train()  # Fine-tune BERT on your data
model.save_pretrained("bert_noisy_pretrain")


num noisy seqs: 178737
Train sample: {'input_ids': tensor([  101,  1026,  1052, 10790,  2509,  1028,  1026,  2006,  2487,  1028,
         1026,  2125,  2487,  1028,  1026,  2310,  2140, 17914,  1028,  1026,
        27723,  1028,  1026,  1052, 10790,  2509,  1028,  1026,  2006,  2487,
         1028,  1026,  2125,  2487,  1028,  1026,  2310,  2140, 17914,  1028,
         1026, 27723,  1028,  1026,  1052, 27531,  1028,  1026,  2006,  2487,
         1028,  1026,  2125,  2475,  1028,  1026,  2310,  2140, 17914,  1028,
         1026, 27723,  1028,  1026,  1052, 10790,  2509,  1028,  1026,  2006,
         2475,  1028,  1026,  2125,  2475,  1028,  1026,  2310,  2140, 17914,
         1028,  1026, 27723,  1028,  1026,  1052, 14526,  2475,  1028,  1026,
         2006,  2475,  1028,  1026,  2125,  2509,  1028,  1026,  2310,  2140,
        17914,  1028,  1026, 27723,  1028,  1026,  1052,  2620,  2620,  1028,
         1026,  2006,  2509,  1028,  1026,  2125,  2549,  1028,  1026,  2310,
         2140

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
The speedups for torchdynamo mostly come wih GPU Ampere or higher and which is not detected here.
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker num

Epoch,Training Loss,Validation Loss
0,1.396500,1.432721


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:617: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
# SAVE PRETRAINED MODEL
import shutil
from google.colab import files


# ZIP the model directory
shutil.make_archive("bert_noisy_pretrain", 'zip', "bert_noisy_pretrain")

# Download the ZIP file
files.download("bert_noisy_pretrain.zip")

print("Model zipped and downloaded as bert_noisy_pretrain.zip")


In [ ]:
# HOW TO LOAD THE PRETRAINED MODEL:

import shutil
from transformers import BertForTokenClassification

# Unzip the saved model
shutil.unpack_archive("bert_noisy_pretrain.zip", "bert_noisy_pretrain")

# Load the model with pretrained weights
model = BertForTokenClassification.from_pretrained("bert_noisy_pretrain")

print("Model loaded successfully from bert_noisy_pretrain.zip")


In [ ]:
# Function to count model parameters
def count_model_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")

# Print parameters for the BERT model
count_model_parameters(model)


# Fine-tuning Loop

In [ ]:
from sklearn.model_selection import train_test_split

# Suppose you have high_quality_sequences => list of (X_list, Y_list)
train_clean, temp_clean = train_test_split(high_quality_sequences, test_size=0.2, random_state=42)
val_clean, test_clean  = train_test_split(temp_clean, test_size=0.5, random_state=42)

print(f"Train set size: {len(train_clean)}")
print(f"Validation set size: {len(val_clean)}")
print(f"Test set size: {len(test_clean)}")  # For final model comparison

# Tokenizer & Dataset Preparation
train_clean_dataset = BertFingeringDataset(train_clean, tokenizer=bert_tokenizer, max_length=256)
val_clean_dataset   = BertFingeringDataset(val_clean,   tokenizer=bert_tokenizer, max_length=256)
test_clean_dataset  = BertFingeringDataset(test_clean,  tokenizer=bert_tokenizer, max_length=256)

# Load fine-tuning model from pretrained checkpoint
model_finetune = BertForTokenClassification.from_pretrained("bert_noisy_pretrain", num_labels=num_finger_labels)

# Training arguments
fine_tune_args = TrainingArguments(
    output_dir="bert_finetuned_clean",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=5e-5,  # Lower LR for fine-tuning
    weight_decay=0.01,
    push_to_hub=False,
    report_to="none"
)

# Trainer (Only train + validation; test set is left out)
trainer_finetune = Trainer(
    model=model_finetune,
    args=fine_tune_args,
    train_dataset=train_clean_dataset,
    eval_dataset=val_clean_dataset,
)

# Fine-tune on high-quality dataset
trainer_finetune.train()
model_finetune.save_pretrained("bert_finetuned_clean")




# TEST_CLEAN_DATASET IS OUR ULTIMATE EVAL, REMEMBER TO SET THE SEED TO 42!!!
# Evaluate on the **test set** (final performance comparison)
test_results = trainer_finetune.evaluate(test_clean_dataset)
print("Test Set Evaluation:", test_results)


# To Compare Model Performance
- Set the EXACT SAME train-test-split as here, with the same random seed
- train the smaller RNN/LSTM/Transformer with the same pretraining and fine-tuning paradigm, but with smaller parameters
- compare final test set accuracy on RNN/LSTM/Transformer, in addition to all of the Nathan Metrics


# Other Steps
- setup a cool live demo, ideally with some visualizations to back it up (piano note graphics/highlights)
- do a live demo DURING THE PROJECT SHOWCASE
- look at the ATTENTION SCORE PATTERN of a transformer model to understand what notes influence other notes the most!

In [27]:
import torch
import pandas as pd
import numpy as np
import os

def predict_piece_fingerings_multi_bert(
    model,
    file_path,
    bert_tokenizer,
    pitch_to_idx,
    finger_to_idx=None,
    idx_to_finger=None,
    max_length=512,
    device=None
):
    """
    Reads a fingering file, converts each note into a "word" token, and runs a forward pass
    on a BERT token-classification model. Returns results in the same format as your RNN
    predict_piece_fingerings_multi function:

      (preds, y_true, y_pred_labels, y_true_labels, acc)

    Where:
      - preds: np.array of predicted finger ints (one per note)
      - y_true: np.array of ground truth finger ints (one per note)
      - y_pred_labels: list of predicted finger strings
      - y_true_labels: list of ground truth finger strings
      - acc: float accuracy (0-100)
    """

    # 1) LOAD & PARSE THE FILE
    if not os.path.exists(file_path):
        print(f"File {file_path} does not exist. Skipping.")
        return ([], [], [], [], 0.0)

    df = pd.read_table(
        file_path,
        sep="\t",
        skiprows=1,
        names=[
            "noteID",
            "onset_time",
            "offset_time",
            "spelled_pitch",
            "onset_velocity",
            "offset_velocity",
            "channel",
            "finger_number"
        ]
    )
    if len(df) == 0:
        return ([], [], [], [], 0.0)

    df["spelled_pitch_int"] = df["spelled_pitch"].map(pitch_to_idx).fillna(0).astype(int)
    if finger_to_idx is not None:
        df["finger_int"] = df["finger_number"].astype(str).map(finger_to_idx).fillna(0).astype(int)
        y_true = df["finger_int"].values
    else:
        y_true = None

    seq_len = len(df)  # number of notes
    X_array = np.zeros((seq_len, 6), dtype=np.float32)
    X_array[:, 0] = df["spelled_pitch_int"].astype(float)
    X_array[:, 1] = df["onset_time"].astype(float)
    X_array[:, 2] = df["offset_time"].astype(float)
    X_array[:, 3] = df["onset_velocity"].astype(float)
    X_array[:, 4] = df["offset_velocity"].astype(float)
    X_array[:, 5] = df["channel"].astype(float)

    # 2) CONVERT EACH NOTE → "WORD" TOKEN
    note_tokens = []
    for row in X_array:
        pitch_int, on_t, off_t, on_v, off_v, ch = row
        token = f"<p{int(pitch_int)}> <on{int(on_t)}> <off{int(off_t)}> <vel{int(on_v)}> <c{int(ch)}>"
        note_tokens.append(token)

    # 3) TOKENIZE + PREDICT
    # If #notes > ~510, we do chunking so we don't truncate.
    chunk_size = min(400, max_length - 2)
    all_pred_ints = []
    all_gt_ints   = []

    # OPTIONAL: device setup
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    start_idx = 0
    while start_idx < seq_len:
        end_idx = min(start_idx + chunk_size, seq_len)

        tokens_chunk = note_tokens[start_idx:end_idx]
        y_chunk      = y_true[start_idx:end_idx] if y_true is not None else None

        # 3a) Tokenize this chunk
        encoding = bert_tokenizer(
            tokens_chunk,
            is_split_into_words=True,
            truncation=True,           # If chunk_size < seq_len, we won't exceed max_length
            max_length=max_length,
            return_tensors="pt"
        )
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)
        token_type_ids = encoding.get("token_type_ids", None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)

        # 3b) Forward pass
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        logits = outputs.logits  # (1, seq_len_in_chunk, num_labels)
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy().tolist()

        # 3c) Subword alignment
        word_ids = encoding.word_ids(batch_index=0)  # list of length = seq_len_in_chunk
        chunk_pred_ints = []
        chunk_gt_ints   = []

        current_word_id = None
        for i, w_id in enumerate(word_ids):
            if w_id is None:
                continue  # ignoring [CLS], [SEP], or padding
            if w_id != current_word_id:
                # first subword of a new word => keep it
                chunk_pred_ints.append(preds[i])
                if y_chunk is not None:
                    chunk_gt_ints.append(y_chunk[w_id])
                current_word_id = w_id
            else:
                # subword => ignore
                continue

        all_pred_ints.extend(chunk_pred_ints)
        all_gt_ints.extend(chunk_gt_ints)
        start_idx = end_idx

    # Convert to arrays
    preds = np.array(all_pred_ints, dtype=int)
    y_true_final = np.array(all_gt_ints, dtype=int) if y_true is not None else np.array([])

    # 4) CHECK LENGTH MATCH
    # We expect "preds" to match the total number of notes (seq_len).
    # If not, it means we truncated or something else went wrong.
    if preds.shape[0] != seq_len:
        print(
            f"WARNING: Mismatch in predicted labels. "
            f"Expected {seq_len}, got {preds.shape[0]}. Possibly max_length too small?"
        )

    # 5) MAP TO STRING LABELS
    if idx_to_finger is not None:
        y_pred_labels = [idx_to_finger.get(int(p), "UNK") for p in preds]
    else:
        y_pred_labels = preds.tolist()

    if y_true is not None and idx_to_finger is not None:
        y_true_labels = [idx_to_finger.get(int(g), "UNK") for g in y_true_final]
    else:
        y_true_labels = []

    # 6) ACCURACY
    if len(y_true_final) > 0 and y_true_final.shape[0] == preds.shape[0]:
        from sklearn.metrics import accuracy_score
        acc = accuracy_score(y_true_final, preds) * 100.0
    else:
        acc = 0.0

    return preds, y_true_final, y_pred_labels, y_true_labels, acc


In [28]:
from transformers import BertForTokenClassification, BertTokenizerFast

# Load your BERT model
bert_model = BertForTokenClassification.from_pretrained("bert_finetuned_clean", num_labels=num_finger_labels)
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Then in your evaluation loop:
all_predicted_fingerings = []
all_ground_truth_fingerings = []
all_ids = []
all_lengths = []

for (piece_id, annotator_id), gt_fingering, length in zip(piece_ids, ground_truth_fingerings, lengths):
    file_name = f"{piece_id}-{annotator_id}_fingering.txt"
    file_path = os.path.join(EVAL_DATASET_PATH, file_name)
    if not os.path.exists(file_path):
        print(f"File {file_path} does not exist. Skipping.")
        continue

    # BERT-based prediction
    preds, y_true_vals, pred_labels, true_labels, piece_acc = predict_piece_fingerings_multi_bert(
        model=bert_model,
        file_path=file_path,
        bert_tokenizer=bert_tokenizer,
        pitch_to_idx=pitch_to_int_mapping,
        finger_to_idx=finger_to_int_mapping,
        idx_to_finger=int_to_finger_mapping,
        max_length=512  # or 256 if you prefer chunking
    )

    all_predicted_fingerings.append(preds.tolist())
    all_ground_truth_fingerings.append(y_true_vals.tolist())
    all_ids.append((piece_id, annotator_id))
    all_lengths.append(length)

# Evaluate with your FingeringEvaluator or custom logic
results = proj_consts_.evaluate_fingering_method(
    predicted_fingerings=all_predicted_fingerings,
    ground_truth_fingerings=all_ground_truth_fingerings,
    piece_ids=all_ids,
    lengths=all_lengths,
    method_name="BERT Model",
    evaluator=evaluator
)

evaluator.print_results(results, method_name="BERT Model")


KeyboardInterrupt: 